In [2]:
import pandas as pd

HARRIER_CSV = '../../queries/sft-train/negatives/harrier-medium-extra.csv'
HARRIER_PASSAGES = '../../dataset/ast/filtered-passages/harrier-medium-filtered-passages.jsonl'
SPLADE_CSV = '../../queries/sft-train/negatives/splade-extra.csv'
SPLADE_PASSAGES = '../../dataset/ast/filtered-passages/splade-filtered-passages.jsonl'
BM25_CSV = '../../queries/sft-train/negatives/bm25-passage-extra.csv'
BM25_PASSAGES = '../../dataset/ast/filtered-passages/bm25-filtered-passages.jsonl'
SELECTED_PASSAGES_JSONL = '../../queries/sft-train/selected-passages.jsonl'

harrier = pd.read_csv(HARRIER_CSV, index_col=None).merge(pd.read_json(HARRIER_PASSAGES, lines=True), how='inner', on='docno')
harrier['retriever'] = 'harrier'
splade = pd.read_csv(SPLADE_CSV, index_col=None).merge(pd.read_json(SPLADE_PASSAGES, lines=True), how='inner', on='docno')
splade['retriever'] = 'splade'
bm25 = pd.read_csv(BM25_CSV, index_col=None).merge(pd.read_json(BM25_PASSAGES, lines=True), how='inner', on='docno')
bm25['retriever'] = 'bm25'

selected_passages = pd.read_json(SELECTED_PASSAGES_JSONL, lines=True).rename(columns={'query_id': 'qid'}).set_index('qid')
selected_passages['num_passages'] = selected_passages['passages'].map(len)

harrier = harrier[harrier['qid'].isin(selected_passages.index)]
splade = splade[splade['qid'].isin(selected_passages.index)]
bm25 = bm25[bm25['qid'].isin(selected_passages.index)]

selected_passages['num_passages'].sum()

6125

In [3]:
def get_top_k(df: pd.DataFrame, k: int=10) -> pd.DataFrame:
    """Extracts the top-k documents (e.g., ranks 1 through 10)."""
    return df[df['rank'] <= k]

def apply_purge(df: pd.DataFrame, k: int=10) -> pd.DataFrame:
    """The Purge: Drops the top-k documents to avoid false negatives."""
    return df[df['rank'] > k]

def false_friends(df: pd.DataFrame, retriever_max_rank: int=35, ce_min_rank: int=100) -> pd.DataFrame:
    """
    Determine false friends from each query. 
    Passages ranked highly by the retriever (e.g., top 30), 
    but scored poorly by the Cross-Encoder (e.g., rank > 100).
    """
    return df[(df['rank_original'] <= retriever_max_rank) & (df['rank'] > ce_min_rank)]

def dynamic_false_friends(group_df: pd.DataFrame, target_n: int, bounds: list[tuple[int,int]] = None):
    if bounds is None:
        bounds = [(35, 100), (40, 90), (45, 80), (50, 70), (60,60), (70,70)]

    pool = pd.DataFrame()
    
    for ret_max, ce_min in bounds:
        pool = group_df[(group_df['rank_original'] <= ret_max) & 
                        (group_df['rank'] > ce_min)]
        
        if len(pool) >= target_n:
            return pool.sample(n=target_n, replace=False)
    
    raise ValueError("Not enough")

def between(df: pd.DataFrame, min_rank: int, max_rank: int) -> pd.DataFrame:
    """
    Determine all documents falling between two ranks (inclusive).
    Used for extracting 'Marginal' negatives.
    """
    return df[(df['rank'] >= min_rank) & (df['rank'] <= max_rank)]

def background_negatives(df: pd.DataFrame, min_rank: int=150) -> pd.DataFrame:
    """
    Extract easy/background negatives.
    Ranked poorly by both the retriever and the CE.
    """
    return df[(df['rank_original'] >= min_rank) & (df['rank'] >= min_rank)]

In [4]:
main_df_params = { #Per selected passage in the target document
    'n_false_friends': 4,
    'n_marginal': 2,
    'n_background': 2,
}

aside_df_params = {
    'n_false_friends': 2,
    'n_marginal': 1,
    'n_background': 1
}

def mine_negatives(df: pd.DataFrame, params: dict):
    'Mines hard negatives from a single sample, for a single query'

    df['query_id'] = df['qid']

    #Make sure top k is cutoff
    df = apply_purge(df)

    ff_sample = df.groupby('qid', group_keys=False).apply(
        lambda x: dynamic_false_friends(
            x, 
            params['n_false_friends'] * selected_passages.loc[x.name, 'num_passages']
        ),
        include_groups=False
    )

    df = df[~df.index.isin(ff_sample.index)]

    marginal_sample = between(df, 26, 60).groupby('qid', group_keys=False).apply(
        lambda x: x.sample(
            n=params['n_marginal'] * selected_passages.loc[x.name, 'num_passages'], 
            replace=False
        ),
        include_groups=False
    )
    df = df[~df.index.isin(marginal_sample.index)]

    background_sample = background_negatives(df).groupby('qid', group_keys=False).apply(
        lambda x: x.sample(
            n=params['n_background'] * selected_passages.loc[x.name, 'num_passages'], 
            replace=False
        ),
        include_groups=False
    )
    
    ff_sample['stratum'] = 'false_friend'
    marginal_sample['stratum'] = 'marginal'
    background_sample['stratum'] = 'background'

    return pd.concat([ff_sample, marginal_sample, background_sample])

def mine_multi_sample_negatives(main_df: pd.DataFrame, main_params: dict, other_dfs: list[pd.DataFrame], other_params):
    'Mines hard negatives from multiple samples'

    main = mine_negatives(main_df, main_params)
    other = pd.concat([mine_negatives(o, other_params) for o in other_dfs])

    return pd.concat([main,other])

train_data = mine_multi_sample_negatives(splade, main_df_params, [harrier, bm25], aside_df_params)
train_data

,Unnamed: 0,Q,docno,rank,score,run_id,rank_original,score_original,text,retriever,query_id,stratum
203,203,Q0,8674423,203,-20.375,splade-passage-prepassage-nemotron-reranked,15,3.713177e+09,5th Asian Film Awards - Section: Nominees and ...,splade,1,false_friend
263,263,Q0,11285768,263,-20.625,splade-passage-prepassage-nemotron-reranked,19,3.695653e+09,Shin Su-won. own experiences as a thirty-somet...,splade,1,false_friend
471,471,Q0,2246525,471,-22.875,splade-passage-prepassage-nemotron-reranked,12,3.732194e+09,Asian Americans in arts and entertainment - Se...,splade,1,false_friend
267,267,Q0,11710413,267,-20.625,splade-passage-prepassage-nemotron-reranked,35,3.579277e+09,Seoul International Women's Film Festival - Se...,splade,1,false_friend
223,223,Q0,11710447,223,-20.500,splade-passage-prepassage-nemotron-reranked,2,3.902590e+09,Seoul International Women's Film Festival - Se...,splade,1,false_friend
...,...,...,...,...,...,...,...,...,...,...,...,...
497676,497676,Q0,3656277,198,-20.375,base-bm25-nemotron-reranked,201,2.203952e+01,Elementals (Marvel Comics). Team lineup Hellf...,bm25,2499,background
497668,497668,Q0,20849714,190,-20.250,base-bm25-nemotron-reranked,227,2.181333e+01,Gosei Sentai Dairanger - Section: Gorma Monste...,bm25,2499,background
498604,498604,Q0,14029655,229,-23.625,base-bm25-nemotron-reranked,225,4.070130e+01,1965 San Diego State Aztecs football team. The...,bm25,2500,background
498544,498544,Q0,17219531,169,-22.625,base-bm25-nemotron-reranked,187,4.111275e+01,1978 Baldwin–Wallace Yellow Jackets football t...,bm25,2500,background


In [5]:
print(selected_passages.head(1)['passages'].iloc[0])

[{'passage': "Vive L'Amour - Section: Plot. Ah-hung to one of the properties she has been trying to sell and has sex with him.  Lee Kang-sheng (李康生) as Hsiao-kang – a salesman for commercial ossuaries (納骨塔), who discovers an apartment key and secretly moves into the apartment.  Chen Chao-jung (陳昭榮) as Ah-jung – a street vendor, who steals the key to the apartment May Lin brings him to and later moves into the apartment. Sharing the absurd life situations together, he forms a friendship with Hsiao-kang after having a quarrel with him at the apartment where they both secretly live in.  Lu Yi-ching (陸弈靜) as a coffee shop owner Reception Vive L'Amour won three Golden Horse Awards, for Best Picture, Best Director, and Best Sound Effects. It also won the Golden Lion award at the 51st Venice International Film Festival. Tsai Ming-liang won Best Director among Edward Yang for A Confucian Confusion, Wong Kar-Wai for Chungking Express, and Stanley Kwan for Red Rose White Rose. He received the aw

In [58]:
false_friends(harrier, ce_min_rank=28).groupby('qid').count().sort_values('score')

,Unnamed: 0,Q,docno,rank,score,run_id,rank_original,score_original,text,retriever
qid,,,,,,,,,,
1541,8,8,8,8,8,8,8,8,8,8
550,9,9,9,9,9,9,9,9,9,9
2004,9,9,9,9,9,9,9,9,9,9
2014,9,9,9,9,9,9,9,9,9,9
919,9,9,9,9,9,9,9,9,9,9
...,...,...,...,...,...,...,...,...,...,...
1312,35,35,35,35,35,35,35,35,35,35
742,35,35,35,35,35,35,35,35,35,35
405,36,36,36,36,36,36,36,36,36,36


In [6]:
import numpy as np
import pandas as pd

STRATUM_MAP = {'false_friend': 0, 'marginal': 1, 'background': 2}
RETRIEVER_MAP = {'harrier': 0, 'splade': 1, 'bm25': 2}

def build_training_dataset(
    golden_df: pd.DataFrame, 
    negatives_df: pd.DataFrame,
) -> list[dict]:
    
    # Create a copy to avoid pandas SettingWithCopyWarning
    negatives_df = negatives_df.copy()
    
    negatives_df['meta'] = list(zip(
        negatives_df['retriever'].map(RETRIEVER_MAP),
        negatives_df['stratum'].map(STRATUM_MAP)
    ))
    
    training_data = []

    for qid, gold_row in golden_df.iterrows():
        golden_passages = gold_row['passages']
        num_golden = len(golden_passages)
        
        # Skip if there are no golden passages to avoid ZeroDivisionError
        if num_golden == 0:
            continue
            
        # Get all negatives for this query
        query_negs = negatives_df[negatives_df['query_id'] == qid]
        
        # 1. Group negatives by both retriever and stratum
        groups = query_negs.groupby(['retriever', 'stratum'])
        
        # 2. Split each group's negatives into `num_golden` roughly equal chunks
        distributed_negs = {}
        for group_key, group in groups:
            records = group.to_dict('records')
            if records:
                # np.array_split returns numpy arrays; we convert them back to lists
                chunks = [chunk.tolist() for chunk in np.array_split(records, num_golden)]
                distributed_negs[group_key] = chunks
        
        # 3. Assemble the final dictionary for each individual golden passage
        for i, gold_pass in enumerate(golden_passages):
            local_negs = []
            
            # Gather the i-th chunk from every (retriever, stratum) combination
            for group_key, chunks in distributed_negs.items():
                if i < len(chunks):
                    local_negs.extend(chunks[i])
                
            training_data.append({
                'qid': qid,
                'passage': gold_pass['passage'], 
                'mined_negatives': [neg['text'] for neg in local_negs],
                'mined_negatives_meta': [neg['meta'] for neg in local_negs]
            })

    return training_data

# Ensure you pass all 5 required positional arguments when calling the function
final = build_training_dataset(
    golden_df=selected_passages, 
    negatives_df=train_data,
)

In [7]:
final_df = pd.DataFrame(final)
final_df

,qid,passage,mined_negatives,mined_negatives_meta
0,1,Vive L'Amour - Section: Plot. Ah-hung to one o...,[Chantal Ughi - Section: Acting career. and ph...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
1,1,Vive L'Amour - Section: Plot. Chao-jung) is ha...,[Thai queer cinema - Section: Sub-genres. unde...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
2,1,Vive L'Amour - Section: Plot. sneak out quietl...,"[Parthan Mohan - Section: Early life. itham’, ...","[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
3,2,Left in Darkness. Left in Darkness is a 2006 h...,[Anando Brahma. son's surgery. Raju is deaf an...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
4,3,Dead Dudes in the House. Dead Dudes in the Hou...,[Takut Ke Tak. Takut Ke Tak () is a 2020 Malay...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
...,...,...,...,...
6120,2499,The Last Airbender (film). The Last Airbender ...,[Elementals (Marvel Comics). Team lineup Hell...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
6121,2499,The Last Airbender (film) - Section: Productio...,[Gosei Sentai Dairanger - Section: Gorma Monst...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
6122,2500,1921 Michigan Wolverines football team. The 19...,[1965 San Diego State Aztecs football team. Th...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."
6123,2500,1921 Michigan Wolverines football team. Americ...,[1978 Baldwin–Wallace Yellow Jackets football ...,"[(2, 2), (2, 0), (2, 0), (2, 1), (0, 2), (0, 0..."


In [9]:
#Almost there, load queries into dataset
QUERIES = '../../queries/sft-train/rewritten-queries-sft.jsonl'
queries = pd.read_json(QUERIES, lines=True)[['query_id', 'BM25_query', 'DENSE_query', 'SPLADE_query']].rename(columns={'query_id': 'qid'})
queries.merge(final_df, how='inner', on='qid').to_json('splade_dataset.jsonl', lines=True, orient='records')